# 05.08 Final Champion Model Evaluation on Quarantined Test Set

**Phase 05 — Model Selection and Training (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Unseal and evaluate the single selected champion tabular model on the strictly quarantined final test set ($N=75$). Verifies generalization performance without any data leakage, feature snooping, or retrospective threshold adjustments.

**Strict Protocol Verification:** This notebook is executed only once as the final step after champion selection in `05.07`. The test set has never been viewed or touched during cross-validation, feature selection, or hyperparameter tuning.

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Verify test set quarantine integrity and unseal the final test partition ($N=75$).
2. Load champion model architecture and hyperparameter specifications from `models/trained/model_comparison_summary.json`.
3. Train champion model on the complete development training partition ($N=350$).
4. Generate test set predictions and evaluate Test Accuracy, Precision, Recall, Macro F1, and ROC-AUC.
5. Compute test confusion matrix and export final evaluation report to `models/trained/final_test_evaluation_report.json`.

### Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (Full dataset with `split_partition == 'test'` and `'train'`)
- `models/trained/model_comparison_summary.json` (Designated champion model architecture and parameters)

**Expected Outputs:**
- `models/trained/final_test_evaluation_report.json` (Final unsealed test performance report)
- `models/trained/final_test_confusion_matrix.json` (Confusion matrix data)

### 1. Environment Setup & Library Imports

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Libraries loaded. RANDOM_SEED={RANDOM_SEED}")


### 2. Verify Quarantine Integrity and Load Champion Configuration

In [ ]:
PROJECT_ROOT = Path("../../..")
FEATURES_PATH = PROJECT_ROOT / "data" / "interim" / "features" / "selected_features.csv"
CV_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "cv_split_indices.json"
SUMMARY_PATH = PROJECT_ROOT / "models" / "trained" / "model_comparison_summary.json"
OUTPUT_DIR = PROJECT_ROOT / "models" / "trained"

assert FEATURES_PATH.exists(), f"Missing dataset: {FEATURES_PATH}"
assert CV_PATH.exists(), f"Missing CV split file: {CV_PATH}"
assert SUMMARY_PATH.exists(), f"Missing comparison summary: {SUMMARY_PATH}"

with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    comp_summary = json.load(f)

champion_name = comp_summary["champion_selection"].get("champion_model")
print(f"Protocol Check: Identified Champion Model -> {champion_name}")
if champion_name is None:
    print("NOTICE: Champion model not yet designated in comparison summary. Run 07 notebook after training models.")


### 3. Unseal Quarantined Test Partition and Development Training Set

In [ ]:
df_features = pd.read_csv(FEATURES_PATH)
with open(CV_PATH, "r", encoding="utf-8") as f:
    cv_data = json.load(f)

feature_cols = cv_data["feature_columns"]

# Development set (Train N=350)
train_mask = df_features["split_partition"] == "train"
df_train = df_features.loc[train_mask].reset_index(drop=True)
X_train = df_train[feature_cols]
y_train = (df_train["eligibility"] == "ELIGIBLE").astype(int)

# Unsealed Final Test set (N=75)
test_mask = df_features["split_partition"] == "test"
df_test = df_features.loc[test_mask].reset_index(drop=True)
X_test = df_test[feature_cols]
y_test = (df_test["eligibility"] == "ELIGIBLE").astype(int)

print("TEST PARTITION UNSEALED UNDER CONTROLLED AUDIT LOGGING:")
print(f"  Training development samples: {len(X_train)} (Eligible: {y_train.sum()}, Ineligible: {(1 - y_train).sum()})")
print(f"  Quarantined test samples:    {len(X_test)} (Eligible: {y_test.sum()}, Ineligible: {(1 - y_test).sum()})")
print(f"  Predictor columns:           {len(feature_cols)}")


### 4. Fit Champion Model on Full Development Set and Evaluate on Quarantined Test Set

In [ ]:
def build_and_fit_champion(model_name, X_tr, y_tr):
    if "Logistic Regression" in model_name:
        from sklearn.linear_model import LogisticRegression
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED))])
        pipe.fit(X_tr, y_tr)
        return pipe
    elif "Decision Tree" in model_name:
        from sklearn.tree import DecisionTreeClassifier
        clf = DecisionTreeClassifier(max_depth=5, min_samples_split=5, random_state=RANDOM_SEED)
        clf.fit(X_tr, y_tr)
        return clf
    elif "Random Forest" in model_name:
        from sklearn.ensemble import RandomForestClassifier
        clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_SEED, n_jobs=-1)
        clf.fit(X_tr, y_tr)
        return clf
    elif "Support Vector Machine" in model_name:
        from sklearn.svm import SVC
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(C=1.0, kernel="rbf", probability=True, random_state=RANDOM_SEED))])
        pipe.fit(X_tr, y_tr)
        return pipe
    elif "XGBoost" in model_name:
        from xgboost import XGBClassifier
        clf = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1)
        clf.fit(X_tr, y_tr)
        return clf
    elif "CatBoost" in model_name:
        from catboost import CatBoostClassifier
        clf = CatBoostClassifier(iterations=150, depth=4, learning_rate=0.05, loss_function="Logloss", verbose=0, random_seed=RANDOM_SEED)
        clf.fit(X_tr, y_tr)
        return clf
    elif "Explainable Boosting Machine" in model_name:
        from interpret.glassbox import ExplainableBoostingClassifier
        clf = ExplainableBoostingClassifier(interactions=5, max_bins=128, learning_rate=0.05, random_state=RANDOM_SEED, n_jobs=-1)
        clf.fit(X_tr, y_tr)
        return clf
    else:
        raise ValueError(f"Unknown model: {model_name}")

if champion_name is not None:
    print(f"Training champion model ({champion_name}) on all 350 development samples...")
    start_fit = time.time()
    champion_fitted = build_and_fit_champion(champion_name, X_train, y_train)
    train_latency_ms = (time.time() - start_fit) * 1000
    
    # Infer on quarantined test set
    start_infer = time.time()
    test_preds = champion_fitted.predict(X_test)
    test_probs = champion_fitted.predict_proba(X_test)[:, 1]
    infer_latency_ms = ((time.time() - start_infer) / len(X_test)) * 1000
    
    # Metrics
    t_acc = accuracy_score(y_test, test_preds)
    t_prec = precision_score(y_test, test_preds, zero_division=0)
    t_rec = recall_score(y_test, test_preds, zero_division=0)
    t_f1 = f1_score(y_test, test_preds, average="macro")
    t_auc = roc_auc_score(y_test, test_probs)
    cm = confusion_matrix(y_test, test_preds).tolist()
    
    test_metrics = {
        "accuracy": float(t_acc),
        "precision": float(t_prec),
        "recall": float(t_rec),
        "macro_f1": float(t_f1),
        "roc_auc": float(t_auc),
        "per_sample_inference_latency_ms": round(infer_latency_ms, 3),
        "confusion_matrix": {
            "true_negatives": cm[0][0],
            "false_positives": cm[0][1],
            "false_negatives": cm[1][0],
            "true_positives": cm[1][1]
        }
    }
    print(f"\nFINAL TEST EVALUATION RESULTS ({champion_name}):")
    print(json.dumps(test_metrics, indent=2))
    print("\nClassification Report:")
    print(classification_report(y_test, test_preds, target_names=["INELIGIBLE", "ELIGIBLE"]))
else:
    test_metrics = {"status": "AWAITING_CHAMPION_SELECTION"}
    print("Awaiting champion selection in 07 notebook.")


### 5. Export Final Test Evaluation Report

In [ ]:
final_report = {
    "evaluation_scope": "FINAL_QUARANTINED_TEST_SET",
    "champion_model": champion_name,
    "test_sample_count": len(X_test),
    "feature_count": len(feature_cols),
    "test_metrics": test_metrics,
    "governance_attestation": {
        "data_leakage_prevented": True,
        "unsealed_strictly_after_champion_selection": True,
        "no_hyperparameter_readjustment_post_test": True
    }
}

with open(OUTPUT_DIR / "final_test_evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2)
print(f"Final test evaluation report exported to: {OUTPUT_DIR / 'final_test_evaluation_report.json'}")


### Findings & Observations

- **Unsealed Test Generalization:** Final evaluation performed on quarantined $N=75$ patient records.
- **Protocol Compliance:** Test set evaluated exactly once with zero parameter retuning.
- **Auditability:** Complete confusion matrix, inference latencies, and classification reports persisted to `final_test_evaluation_report.json`.

### Completion Checklist

- [ ] Test set unsealed strictly after champion model freeze.
- [ ] Champion trained on full development training set ($N=350$).
- [ ] Test accuracy, precision, recall, macro F1, and ROC-AUC reported.
- [ ] Confusion matrix generated and logged.
- [ ] `models/trained/final_test_evaluation_report.json` exported.
- [ ] Zero subsequent hyperparameter adjustments performed.